# 01 · Style definition and success metric

This notebook settles the two definitions that everything else depends on:

1. **What is a style?** The unit we predict. Each style needs a stable, unique `style_id` for the API.
2. **What makes a style successful?** The quantity the model will learn to predict.

Each decision comes with the evidence from the data that supports it.

**Data used:** `articles.csv` and `transactions_train.csv`. `customers.csv` isn't required by the brief;
customer behaviour can be derived from `customer_id` in transactions.

In [ ]:
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data_science").is_dir())
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from data_science.data import load_articles, load_transactions
from data_science.features.style import STYLE_ATTRIBUTES, GROUP_ATTRIBUTES, add_style_columns, make_id
from data_science.features.success import success_scores, MIN_ARTICLES

pd.set_option("display.width", 200, "display.max_columns", 30, "display.max_colwidth", 80)

In [ ]:
articles = load_articles()
tx = load_transactions(columns=["t_dat", "article_id", "price"])
print(f"articles:     {len(articles):,} rows")
print(f"transactions: {len(tx):,} rows, {tx.t_dat.min().date()} → {tx.t_dat.max().date()}")

## 1. The tables

### `articles.csv`: one row per article (one product in one colour)

Most attributes come in pairs: a code (`_no` / `_id` / `_code`) and its label (`_name`).

In [ ]:
overview = pd.DataFrame({
    "unique values": articles.nunique(),
    "missing": articles.isna().sum(),
    "example": articles.iloc[0],
})
overview

### `transactions_train.csv`: one row per unit sold

There is no quantity column, so each row is one unit. `price` is scaled by Kaggle (not a currency), so
only comparisons between prices are meaningful. Stock, returns and promotions aren't available.

In [ ]:
print(f"articles sold at least once: {tx.article_id.nunique():,} of {len(articles):,}")
tx.price.describe().to_frame("price").T

## 2. Defining a style

A style is the set of attributes that **together define a customer's style choice**. It has to be:

- **Exhaustive**: changing any attribute *outside* the set doesn't change the style.
- **Non-redundant**: changing the value of any attribute *inside* the set gives a different style.

### 2.1 Which columns duplicate others?

If column A determines column B (each A value maps to one B value), B adds no information once A is included.

In [ ]:
def determines(a, b):
    n_values = articles.groupby(a)[b].nunique()
    return f"{(n_values > 1).sum()} of {len(n_values)} {a} values map to >1 {b}"

pairs = [("product_type_name", "product_group_name"),
         ("index_name", "index_group_name"),
         ("colour_group_name", "perceived_colour_master_name"),
         ("colour_group_name", "perceived_colour_value_name"),
         ("graphical_appearance_no", "graphical_appearance_name")]
for a, b in pairs:
    print(determines(a, b))

- `product_type` → `product_group`: determined (1 exception), so **`product_group` is a duplicate**.
- `index` → `index_group`: `index_group` is a coarser version. The extra splits in `index` (accessories, lingerie) are already captured by product type.
- `graphical_appearance_no` ↔ `_name`: one-to-one, so they're interchangeable.
- `colour_group` does **not** determine colour tone, so tone may add information. Checked below.

### 2.2 Column-by-column verdict

| Column | Verdict | Reason |
|---|---|---|
| `article_id`, `product_code`, `prod_name` | ❌ identifiers | name a product, not a style |
| `product_group_name` | ❌ duplicate | determined by product type |
| `index_name` | ❌ duplicate | finer version of index group; extra splits covered by product type |
| `perceived_colour_master_name` | ❌ duplicate | colour family; `colour_group_name` is finer |
| `department_name`, `section_name` | ❌ internal organisation | buying teams and store layout; customers don't see them |
| `garment_group_name` | ❌ unreliable | mixes fabric, type and labels like "Special Offers"; many "Unknown" |
| **`index_group_name`** | ✅ who it is for | Ladieswear / Menswear / Divided / Baby-Children / Sport |
| **`product_type_name`** | ✅ what it is | dress, trousers, hoodie, ... |
| **`graphical_appearance_name`** | ✅ surface design | pattern and texture |
| **`colour_group_name`** | ✅ colour | 50 specific colours |
| **`perceived_colour_value_name`** | ✅ tone | dark / light / dusty light / bright, ... |

### 2.3 Does colour tone add anything beyond the colour itself?

In [ ]:
tone = pd.crosstab(articles.colour_group_name, articles.perceived_colour_value_name, normalize="index")
colours = ["Black", "White", "Dark Blue", "Light Pink", "Red", "Blue", "Pink", "Yellow", "Green"]
tone.loc[colours].round(2)

In [ ]:
dominant_share = (articles.groupby("colour_group_name").perceived_colour_value_name
                  .agg(lambda v: v.value_counts().iloc[0]).sum() / len(articles))
print(f"articles whose tone is the most common tone for their colour: {dominant_share:.1%}")

For most colours the tone is fixed (Black → Dark, White → Light), so it adds nothing there and does no harm.
Where it varies, the split is meaningful: **Red is split between Medium and Bright** (classic vs neon red), and
**Light Pink between Dusty Light and Light** (muted vs clean pastel). Tone stays in.

### 2.4 Is "who it is for" needed?

Without `index_group_name`, would men's, women's and kids' products end up in the same style?

In [ ]:
no_segment = [c for c in STYLE_ATTRIBUTES if c != "index_group_name"]
key = make_id(articles, no_segment)
segments_per_style = articles.groupby(key).index_group_name.nunique()
mixed = segments_per_style[segments_per_style > 1].index
print(f"styles without segment: {key.nunique():,}")
print(f"styles mixing >1 customer segment: {len(mixed):,}, "
      f"covering {key.isin(mixed).mean():.0%} of articles")

examples = (articles[key.isin(mixed)].assign(style=key)
            .groupby("style").index_group_name.value_counts().unstack(fill_value=0))
examples.loc[examples.sum(axis=1).nlargest(5).index]

Most articles would sit in styles that mix customer segments. A parent buying kids' trousers would never
consider men's trousers instead, so segment changes the purchase decision. **It is part of the style.**

### 2.5 Final style definition

Data-quality fix: `product_type_name` contains both "Dog Wear" and "Dog wear". Values that differ only in
case are merged to their most common spelling, so the same style can't get two names.

In [ ]:
styled = add_style_columns(articles)

assert styled.groupby("style_id").style_name.nunique().max() == 1
assert styled.groupby("style_name").style_id.nunique().max() == 1

per_style = styled.groupby("style_id").size()
print(f"styles: {styled.style_id.nunique():,}   groups (index_group + product_type): {styled.group_id.nunique()}")
print(f"articles per style: median {per_style.median():.0f}, mean {per_style.mean():.1f}, max {per_style.max()}")
styled[["article_id", *STYLE_ATTRIBUTES, "style_id"]].head(3)

`style_id` is a readable slug, e.g. `ladieswear__dress__all-over-pattern__light-pink__dusty-light`. It is
unique, it's built only from article attributes (so it doesn't change when the model is retrained), and it is URL-safe.

### 2.6 Why not articles or product codes?

Next season's products mostly don't exist yet. How many of the articles and styles that sold in one season
also sold in the same season a year later?

In [ ]:
def sold_in(start, end):
    w = tx[(tx.t_dat >= start) & (tx.t_dat <= end)]
    return set(w.article_id.unique())

style_of = styled.set_index("article_id").style_id
a18, a19 = sold_in("2018-09-23", "2018-12-15"), sold_in("2019-09-23", "2019-12-15")
s18, s19 = set(style_of.loc[list(a18)]), set(style_of.loc[list(a19)])
print(f"Autumn 2019 articles that also sold in Autumn 2018: {len(a19 & a18) / len(a19):.0%}")
print(f"Autumn 2019 styles   that also sold in Autumn 2018: {len(s19 & s18) / len(s19):.0%}")

w19 = tx[(tx.t_dat >= "2019-09-23") & (tx.t_dat <= "2019-12-15")]
rev_old_articles = w19.loc[w19.article_id.isin(a18), "price"].sum() / w19.price.sum()
rev_old_styles = w19.loc[w19.article_id.map(style_of).isin(s18), "price"].sum() / w19.price.sum()
print(f"Autumn 2019 revenue from articles that existed in Autumn 2018: {rev_old_articles:.0%}")
print(f"Autumn 2019 revenue from styles   that existed in Autumn 2018: {rev_old_styles:.0%}")

Most articles are new each season, but almost all revenue comes from styles that already existed a year
earlier. The styles that don't recur are mostly tiny. That makes styles the right unit for a *next-season* prediction.

### 2.7 Known gap

Fit, length, neckline and fabric clearly affect a customer's choice, but they only exist in the free-text
`detail_desc` (and in images), not in structured columns. Extracting them would push the number of styles
towards one style per product, which brings back the sparsity and turnover problem. They are left out and noted as a limitation.

In [ ]:
articles.loc[articles.product_type_name.isin(["Dress", "Trousers"]), ["product_type_name", "detail_desc"]].sample(4, random_state=1)

## 3. Defining a successful style

The goal isn't to find the best-selling products. It's to find **which styles perform best**. The metric should:

- reward **revenue** (how much the style earned),
- penalise **discounting** (selling at full price shows real demand),
- remove the **inherent demand of the category** (trousers sell more than gloves whatever their style),
- **not depend on how many styles a category has.**

### 3.1 The formula (per style, per 12-week season)

```
Adjusted revenue = Revenue × (Revenue ÷ Full-price revenue)
Group benchmark  = Σ Adjusted revenue² ÷ Σ Adjusted revenue    (over styles in the same index_group + product_type)
Success score    = Adjusted revenue ÷ Group benchmark
```

- **Full-price revenue** is what the units would have earned at each article's highest observed price.
- **Group benchmark** is the revenue-weighted typical style size of the category: each style counts in
  proportion to its revenue, so a long tail of tiny styles barely moves it.
- **A score of 1** means "as good as the typical style a sale in this category comes from"; 2 means twice that.
- **Minimum-size floor:** a style needs at least `MIN_ARTICLES` active articles to be eligible to win.

### 3.2 Is the highest price a safe "full price"?

In [ ]:
p = tx.groupby("article_id").price.agg(["max", "median"])
(p["max"] / p["median"]).quantile([.5, .9, .99]).rename("max ÷ median price per article").to_frame().T.round(2)

For a typical article the highest price equals its median price, and even at the 99th percentile it's
under 3×. There are no extreme outliers, so the maximum is a reasonable estimate of the undiscounted price.

### 3.3 Scores on three past seasons

In [ ]:
SEASONS = {
    "Summer 2019": ("2019-06-30", "2019-09-22"),
    "Autumn 2019": ("2019-09-23", "2019-12-15"),
    "Summer 2020": ("2020-06-30", "2020-09-22"),
}
scores = {name: success_scores(tx, styled, *window) for name, window in SEASONS.items()}
for name, s in scores.items():
    print(f"{name}: {len(s):,} styles sold, {s.eligible.sum():,} eligible (≥{MIN_ARTICLES} articles), "
          f"covering {s.loc[s.eligible, 'revenue'].sum() / s.revenue.sum():.0%} of revenue")

In [ ]:
cols = ["style_name", "success_score", "full_price_factor", "revenue", "units", "articles", "group_styles"]
s = scores["Summer 2020"]
print("Summer 2020: top 15 eligible styles by success score")
s[s.eligible][cols].head(15).round(2)

In [ ]:
print("Summer 2020: top 10 by raw revenue, for comparison")
s.nlargest(10, "revenue")[cols].round(2)

Raw revenue puts only Ladieswear basics at the top. The success score spreads the leaders across categories
and customer segments.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3))
ax.hist(s.loc[s.eligible, "full_price_factor"], bins=40, color="#4C72B0")
ax.set(title="Full-price factor of eligible styles (Summer 2020)", xlabel="revenue ÷ full-price revenue", ylabel="styles")
plt.show()

### 3.4 Is the score independent of the number of styles in a category?

Revenue **share** (style revenue ÷ category revenue) was rejected because shares in a category always add
up to 100%: a category with many styles gives each style a smaller share. Below: the top score in each
category against the category's number of styles (categories with ≥5 styles).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 3.5))
for ax, (name, s_) in zip(axes, scores.items()):
    s_ = s_.assign(share=s_.revenue / s_.groupby("group_id").revenue.transform("sum"))
    g = s_.groupby("group_id").agg(n=("group_styles", "first"), top_score=("success_score", "max"), top_share=("share", "max"))
    g = g[g.n >= 5]
    r_score = g.top_score.corr(g.n, method="spearman")
    r_share = g.top_share.corr(g.n, method="spearman")
    print(f"{name}: Spearman with #styles: top success score {r_score:+.2f}, top share {r_share:+.2f}")
    ax.scatter(g.n, g.top_score, s=10)
    ax.set(xscale="log", title=f"{name}\nSpearman: score {r_score:+.2f} | share {r_share:+.2f}",
           xlabel="styles in category", ylabel="top success score")
plt.tight_layout(); plt.show()

Share is strongly *negatively* related to the number of styles, as expected. The success score is
*positively* related: a dominant basic stands out more against a category full of mid-sized styles.
Neither is fully independent. The success score is kept because it works on the same scale across categories
(revenue-weighted average = 1 in every category). This is recorded as a limitation.

### 3.5 Why the minimum-size floor is needed

In [ ]:
for name, s_ in scores.items():
    top = s_.head(50)
    print(f"{name}: {(~top.eligible).sum()} of the top 50 have fewer than {MIN_ARTICLES} articles")
print("
Summer 2019: top 10 with no floor")
scores["Summer 2019"][cols + ["eligible"]].head(10).round(2)

Without a floor, styles with only 1–2 articles can rank near the top, e.g. a Baby/Children jumpsuit with
2 articles and 156 units in Summer 2019. Their categories are small, so the benchmark is small too.
Requiring at least `MIN_ARTICLES` active articles removes them.

The floor doesn't remove small-revenue styles that have enough articles: several Baby/Children styles
(13–38 articles, a few hundred units) still rank highly in Summer 2020. That's intended. They outperform
their own category, and kids' categories simply earn less.

### 3.6 Alternative tested and rejected: leave-one-out benchmark

The benchmark includes the style itself, so a strong style raises its own benchmark (a style that
dominates its category scores ≈ 1). A leave-one-out benchmark, built only from the *other* styles, removes this.

In [ ]:
def leave_one_out(s_):
    a = s_.adjusted_revenue
    g = s_.groupby("group_id").adjusted_revenue
    sum_a, sum_a2 = g.transform("sum"), g.transform(lambda x: (x ** 2).sum())
    bench = ((sum_a2 - a ** 2) / (sum_a - a)).where(s_.group_styles > 1)
    return s_.assign(loo_score=a / bench)

loo = leave_one_out(scores["Summer 2020"])
single = loo[loo.group_styles == 1]
print(f"single-style categories (cannot be scored): {len(single)}, {single.revenue.sum() / loo.revenue.sum():.2%} of revenue")
print("\nunfiltered top 5 by leave-one-out score:")
display(loo.nlargest(5, "loo_score")[["style_name", "loo_score", "revenue", "articles", "group_styles"]].round(1))
f = loo[(loo.group_styles >= 10) & loo.eligible]
print("with floors (≥10 styles in category, ≥5 articles):")
f.nlargest(10, "loo_score")[["style_name", "loo_score", "revenue", "articles", "group_styles"]].round(1)

Leave-one-out gives extreme scores (> 100) in categories with only 2–3 styles. Even with floors, the winners
become "default colour" categories: black ballerinas, leggings and tights, where customers barely buy any other
colour. **The inclusive benchmark is kept**, and the self-inclusion effect is noted as a limitation.

### 3.7 Is it predictable?

In [ ]:
def compare(a, b, k=50):
    x = scores[a].loc[scores[a].eligible, ["style_id", "success_score"]]
    y = scores[b].loc[scores[b].eligible, ["style_id", "success_score"]]
    m = x.merge(y, on="style_id", suffixes=("_a", "_b"))
    overlap = len(set(x.head(k).style_id) & set(y.head(k).style_id))
    return {"from": a, "to": b, "common styles": len(m),
            "Spearman": round(m.success_score_a.corr(m.success_score_b, method="spearman"), 2),
            f"top-{k} overlap": f"{overlap}/{k}"}

pd.DataFrame([compare("Summer 2019", "Autumn 2019"), compare("Summer 2019", "Summer 2020"),
              compare("Autumn 2019", "Summer 2020")])

Scores carry over from season to season to a moderate degree: enough persistence for a model to learn from,
and enough change that simply repeating last season isn't the whole answer.

### 3.8 What kind of styles win?

In [ ]:
rows = []
for name, s_ in scores.items():
    top = s_[s_.eligible].head(50)
    rows.append({"season": name, "categories in top 50": top.group_id.nunique(),
                 "Solid": top.style_name.str.contains("· Solid ·").sum(),
                 "Black": top.style_name.str.contains("· Black ·").sum()})
pd.DataFrame(rows)

⚠️ **Supply vs. demand.** The winners are mostly solid black and white basics. Part of that comes from H&M's
own supply decisions: H&M offers far more articles in these styles (e.g. ~100 black solid skirts against a handful in most
other colours), so they naturally earn more in total. The score therefore partly reflects H&M's buying plan,
not only customer preference. Without stock and assortment data we can't fully separate the two. We accept
this deliberately: these styles really did perform best, and predicting what performs is the goal.

## 4. Decisions

| Decision | Choice |
|---|---|
| Style | `index_group + product_type + graphical_appearance + colour_group + perceived_colour_value` |
| `style_id` | slug of the five attributes, e.g. `ladieswear__dress__all-over-pattern__light-pink__dusty-light` |
| Success score | discount-adjusted revenue ÷ revenue-weighted category benchmark (style included) |
| Eligibility | at least 5 active articles in the season |
| Known limitations | no fit/length/fabric in the style; score partly reflects supply; style counts in its own benchmark; score is not fully independent of category size |

**Next:** choose the prediction period.